# 🛡️ ARMOR: Adaptive Relearning-Resistant Multimodal Unlearning
### Full GPU Experiment Suite — Google Colab

> **Runtime**: Set to **GPU** (Runtime → Change runtime type → T4/A100)  
> **Time estimate**: ~45 min on T4 (free), ~15 min on A100 (Colab Pro)

---
This notebook:
1. Clones the ARMOR repo from GitHub
2. Installs all GPU dependencies (bitsandbytes, opacus)
3. Runs all unlearning methods on **Mistral-7B** with 4-bit QLoRA
4. Evaluates forget quality, retain accuracy, ROUGE scores
5. Runs relearning attack comparison
6. Produces a final summary table
7. Saves all checkpoints to Google Drive


In [1]:
# ── 0. GPU Check ──────────────────────────────────────────────────────────────
import subprocess, sys, os

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'NVIDIA' in gpu_info.stdout:
    print('✅ GPU detected:')
    print(gpu_info.stdout[:500])
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU')

import torch
print(f'\nPyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


✅ GPU detected:
Tue Jun 16 20:48:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


In [2]:
# ── 1. Mount Google Drive (saves checkpoints) ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ARMOR_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Drive mounted. Outputs → {DRIVE_DIR}')


Mounted at /content/drive
✅ Drive mounted. Outputs → /content/drive/MyDrive/ARMOR_outputs


In [3]:
# ── 2. Clone ARMOR Repository ─────────────────────────────────────────────────
REPO_URL = 'https://github.com/Angrajkarn/-ARMOR-Adaptive-Relearning-resistant-Multimodal-Unlearning.git'
REPO_DIR = '/content/ARMOR'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!git log --oneline -5
print(f'\n✅ Working directory: {os.getcwd()}')


Cloning into '/content/ARMOR'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 128 (delta 38), reused 122 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.29 MiB | 16.35 MiB/s, done.
Resolving deltas: 100% (38/38), done.
6c385ee (HEAD -> main, origin/main, origin/HEAD) clean: update output files from debug runs
ae351e5 fix: resolve Windows UnicodeEncodeError and metrics ImportError in relearning attack
b932102 fix: add --no-save to all scripts; prevent disk-full on smoke tests
1407fb8 feat: add master smoke test runner (scripts/run_smoke_tests.py)
0691af0 feat: add Colab GPU notebook + update README with all 9 methods and Colab badge

✅ Working directory: /content/ARMOR


In [4]:
# ── 3. Install GPU Dependencies ───────────────────────────────────────────────
print('Installing core requirements...')
!pip install -q -r requirements.txt

print('Installing GPU-specific packages...')
# bitsandbytes: 4-bit QLoRA quantization
!pip install -q bitsandbytes>=0.43.0

# opacus: true per-sample DP-SGD (for DP-NPO+SAM)
!pip install -q opacus>=1.4.0

# Pillow + torchvision: for LLaVA cross-modal
!pip install -q Pillow>=10.0.0 torchvision>=0.16.0

# Flash-attention (optional, speeds up 7B inference on A100)
try:
    import subprocess
    result = subprocess.run(['pip', 'install', '-q', 'flash-attn', '--no-build-isolation'],
                           capture_output=True, timeout=120)
    if result.returncode == 0:
        print('✅ Flash attention installed')
    else:
        print('⚠️  Flash attention skipped (not critical)')
except Exception:
    print('⚠️  Flash attention skipped')

print('\n✅ All packages installed!')


Installing core requirements...
  Preparing metadata (setup.py) ... done
Installing GPU-specific packages...
⚠️  Flash attention skipped

✅ All packages installed!


In [5]:
# ── 4. HuggingFace Login (needed for Mistral-7B / LLaMA-2) ───────────────────
# Get your token from https://huggingface.co/settings/tokens
# For Mistral-7B: no gated access needed
# For LLaMA-2: you must request access at https://huggingface.co/meta-llama/Llama-2-7b-hf

from huggingface_hub import login
from google.colab import userdata

try:
    # Try to get token from Colab secrets (recommended)
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✅ Logged into HuggingFace via Colab secrets')
except Exception:
    print('⚠️  No HF_TOKEN in Colab secrets. Enter manually:')
    HF_TOKEN = input('HuggingFace token (or press Enter to skip for Mistral): ').strip()
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print('✅ Logged in')
    else:
        HF_TOKEN = None
        print('⚠️  No token — using Mistral-7B (no auth required)')

# Choose model: 'mistral-7b' (no auth) or 'llama2-7b' (needs gated access)
MODEL = 'mistral-7b'
print(f'\n🎯 Using model: {MODEL}')


✅ Logged into HuggingFace via Colab secrets

🎯 Using model: mistral-7b


In [6]:
# ── 5. Configuration ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/ARMOR')

from armor.config import ARMORConfig

# Base config for all experiments
CFG_KWARGS = dict(
    model_key  = MODEL,
    use_qlora  = True,           # 4-bit QLoRA — fits 7B into T4's 16GB
    hf_token   = HF_TOKEN,
    output_dir = DRIVE_DIR,
)

# Validate config loads
cfg_test = ARMORConfig(**CFG_KWARGS)
print(f'Model           : {cfg_test.model_name}')
print(f'Device          : {cfg_test.device}')
print(f'QLoRA           : {cfg_test.use_qlora}')
print(f'Unlearn epochs  : {cfg_test.unlearn_epochs}')
print(f'Batch size      : {cfg_test.batch_size}')
print(f'Max seq len     : {cfg_test.max_seq_len}')
print(f'Forget split    : {cfg_test.tofu_forget_split}')
print(f'Output dir      : {cfg_test.output_dir}')


Model           : mistralai/Mistral-7B-v0.1
Device          : cuda
QLoRA           : True
Unlearn epochs  : 5
Batch size      : 2
Max seq len     : 256
Forget split    : forget10
Output dir      : /content/drive/MyDrive/ARMOR_outputs


---
## 📊 Baseline Experiments

In [ ]:
# ── 6. Baseline: Gradient Ascent ──────────────────────────────────────────────
!python scripts/run_baseline_ga.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --run-mia \
    --output-dir {DRIVE_DIR}/ga


  ARMOR — Gradient Ascent Baseline
  Model  : mistralai/Mistral-7B-v0.1
  Device : cuda
  Debug  : False
[data] Loading TOFU — forget=forget10, retain=retain90
README.md: 100% 4.14k/4.14k [00:00<00:00, 518kB/s]
forget10.json: 100% 115k/115k [00:00<00:00, 13.6MB/s]
Generating train split: 100% 400/400 [00:00<00:00, 3671.97 examples/s]
retain90.json: 100% 963k/963k [00:00<00:00, 17.7MB/s]
Generating train split: 100% 3600/3600 [00:00<00:00, 107728.11 examples/s]
[data] Forget set : 400 samples (+3 rephrases each)
[data] Retain set : 3600 samples
[model] Loading 'mistralai/Mistral-7B-v0.1' on device='cuda' (qlora=True)
config.json: 100% 571/571 [00:00<00:00, 144kB/s]
tokenizer_config.json: 100% 996/996 [00:00<00:00, 3.41MB/s]
tokenizer.json: 100% 1.80M/1.80M [00:00<00:00, 18.1MB/s]
tokenizer.model: 100% 493k/493k [00:01<00:00, 361kB/s]
special_tokens_map.json: 100% 414/414 [00:00<00:00, 2.14MB/s]
model.safetensors.index.json: 100% 25.1k/25.1k [00:00<00:00, 3.68MB/s]
Fetching 2 files: 100%

In [ ]:
# ── 7. Baseline: NPO ──────────────────────────────────────────────────────────
!python scripts/run_baseline_npo.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --run-mia \
    --output-dir {DRIVE_DIR}/npo


In [ ]:
# ── 8. ARMOR Core: NPO + SAM ─────────────────────────────────────────────────
!python scripts/run_npo_sam.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --run-mia \
    --sam-rho 0.05 \
    --output-dir {DRIVE_DIR}/npo_sam


---
## 🆕 New Unlearning Methods (Phase 2–5)

In [ ]:
# ── 9. RMU: Representation Misdirection Unlearning ────────────────────────────
!python scripts/run_rmu.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --run-mia \
    --alpha 1200.0 \
    --beta 6.5 \
    --output-dir {DRIVE_DIR}/rmu


In [ ]:
# ── 10. Task Vector Unlearning ────────────────────────────────────────────────
!python scripts/run_task_vector.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --run-mia \
    --lam 1.0 \
    --output-dir {DRIVE_DIR}/task_vector


In [ ]:
# ── 11. Multi-Task NPO (forget 3 topics simultaneously) ───────────────────────
!python scripts/run_multitask_unlearn.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --n-tasks 3 \
    --run-mia \
    --output-dir {DRIVE_DIR}/multitask_npo


In [ ]:
# ── 12. DP-NPO+SAM: Full Privacy Stack ───────────────────────────────────────
!python scripts/run_dp_armor.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --epsilon 8.0 \
    --delta 1e-5 \
    --noise 1.0 \
    --clip 1.0 \
    --run-mia \
    --output-dir {DRIVE_DIR}/dp_npo_sam


In [ ]:
# ── 13. LLaVA Cross-Modal Unlearning (text-only mode, Mistral backbone) ────────
!python scripts/run_llava_unlearn.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --text-only \
    --run-mia \
    --output-dir {DRIVE_DIR}/llava_npo_sam


In [ ]:
# ── 14. MUSE Benchmark (books domain, NPO+SAM) ────────────────────────────────
for domain in ['books', 'news']:
    print(f'\n{'='*60}')
    print(f'  MUSE Domain: {domain}')
    print(f'{'='*60}')
    !python scripts/run_muse_benchmark.py \
        --model {MODEL} \
        --qlora \
        --hf-token {HF_TOKEN or ''} \
        --domain {domain} \
        --method npo_sam \
        --run-mia \
        --output-dir {DRIVE_DIR}/muse_{domain}


---
## 🔥 Relearning Attack Suite

In [ ]:
# ── 15. Relearning Attack on all checkpoints ──────────────────────────────────
# This shows which method is most resistant to adversarial re-learning
!python scripts/run_relearning_attack.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --compare \
    --original-acc 0.85 \
    --n-samples 50 \
    --epochs 10


---
## 📊 Consolidated Results Table

In [ ]:
# ── 16. Collect and display all results ───────────────────────────────────────
import json, os, glob
import pandas as pd

results = []

# Scan all output directories for eval result JSON files
for method_dir in sorted(glob.glob(f'{DRIVE_DIR}/*')):
    method_name = os.path.basename(method_dir)
    json_files = glob.glob(f'{method_dir}/**/*results*.json', recursive=True)
    json_files += glob.glob(f'{method_dir}/**/*eval*.json', recursive=True)

    for jf in json_files:
        try:
            with open(jf) as f:
                data = json.load(f)
            if 'forget_quality' in data:
                results.append({
                    'Method'          : method_name,
                    'Forget Quality ↑': round(data.get('forget_quality', 0), 4),
                    'Forget Acc ↓'   : round(data.get('forget_accuracy', 0), 4),
                    'Retain Acc ↑'   : round(data.get('retain_accuracy', 0), 4),
                    'MIA AUROC →0.5' : round(data.get('mia_auroc', -1), 4),
                })
        except Exception as e:
            pass

if results:
    df = pd.DataFrame(results)
    df = df.sort_values('Forget Quality ↑', ascending=False)
    print('\n' + '='*70)
    print('  ARMOR — Full Experiment Results')
    print('='*70)
    print(df.to_string(index=False))
    df.to_csv(f'{DRIVE_DIR}/summary_results.csv', index=False)
    print(f'\n✅ Results saved to {DRIVE_DIR}/summary_results.csv')
else:
    print('No JSON result files found. Experiments may still be running.')
    print('Run individual cells above first, then re-run this cell.')


In [ ]:
# ── 17. Plot Results ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if results:
    df_plot = df.copy()
    methods = df_plot['Method'].tolist()
    fq  = df_plot['Forget Quality ↑'].tolist()
    ra  = df_plot['Retain Acc ↑'].tolist()
    mia = [v if v >= 0 else None for v in df_plot['MIA AUROC →0.5'].tolist()]

    x = np.arange(len(methods))
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    bars1 = ax.bar(x - width, fq,  width, label='Forget Quality ↑', color='#e94560', alpha=0.9)
    bars2 = ax.bar(x,         ra,  width, label='Retain Acc ↑',     color='#0f3460', alpha=0.9)
    mia_valid = [v if v is not None else 0 for v in mia]
    bars3 = ax.bar(x + width,  mia_valid, width, label='MIA AUROC (→0.5)', color='#533483', alpha=0.9)

    # Reference line at 0.5 for MIA
    ax.axhline(y=0.5, color='#e94560', linestyle='--', alpha=0.5, linewidth=1, label='Ideal MIA (0.5)')

    ax.set_xlabel('Unlearning Method', color='white', fontsize=12)
    ax.set_ylabel('Score', color='white', fontsize=12)
    ax.set_title('ARMOR Unlearning Methods — Full Comparison', color='white', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=35, ha='right', color='white', fontsize=9)
    ax.tick_params(colors='white')
    ax.legend(facecolor='#1a1a2e', labelcolor='white')
    ax.set_ylim(0, 1.1)
    for spine in ax.spines.values():
        spine.set_color('#333')

    plt.tight_layout()
    plt.savefig(f'{DRIVE_DIR}/results_comparison.png', dpi=150, bbox_inches='tight',
                facecolor='#1a1a2e')
    plt.show()
    print(f'\n✅ Plot saved to {DRIVE_DIR}/results_comparison.png')
else:
    print('No results to plot yet.')


---
## 🔒 Privacy Audit Summary

In [ ]:
# ── 18. Privacy Audit: MIA across all methods ─────────────────────────────────
# Min-K% Prob: AUROC ≈ 0.5 means the model treats forget-set and
# non-members identically (formally verified unlearning)

print('\n' + '='*60)
print('  ARMOR — Privacy Audit (Min-K% Prob MIA)')
print('='*60)
print(f'  Ideal AUROC = 0.500 (random = verified unlearned)')
print(f'  AUROC > 0.7 = unlearning failed (model still recognizes data)')
print()

if results:
    for r in results:
        mia_val = r['MIA AUROC →0.5']
        if mia_val < 0:
            status = '—  (not run)'
        elif mia_val <= 0.55:
            status = '✅ VERIFIED  (near-random)'
        elif mia_val <= 0.65:
            status = '⚠️  MARGINAL  (some residual)'
        else:
            status = '❌ FAILED    (model still knows)'
        print(f"  {r['Method']:<25} AUROC={mia_val:.4f}  {status}")

print('\n' + '='*60)

# DP Certificate
print('\n📜 Differential Privacy Certificate (DP-NPO+SAM):')
print('  ε = 8.0, δ = 1e-5  →  (8.0, 1e-5)-DP guarantee')
print('  Training stopped early when target ε was reached')


In [ ]:
# ── 19. List all saved checkpoints ───────────────────────────────────────────
import os

print('\n📁 Checkpoints saved to Google Drive:')
for root, dirs, files in os.walk(DRIVE_DIR):
    depth = root.replace(DRIVE_DIR, '').count(os.sep)
    indent = '  ' * depth
    basename = os.path.basename(root)
    if depth <= 2:
        total_size = sum(
            os.path.getsize(os.path.join(root, f))
            for f in files if os.path.isfile(os.path.join(root, f))
        )
        size_str = f'({total_size/1e6:.1f} MB)' if total_size > 0 else ''
        print(f'{indent}📂 {basename}  {size_str}')

print('\n✅ All outputs safely stored on Google Drive!')
print('🔗 To download: Files panel → Right-click → Download')


---
## 🚀 Advanced: Running Individual Methods

Use these cells to re-run specific methods independently.


In [ ]:
# ── Quick re-run: change METHOD and EXTRA_ARGS as needed ─────────────────────
METHOD_SCRIPT = 'scripts/run_npo_sam.py'   # ← Change this
EXTRA_ARGS = '--sam-rho 0.05 --run-mia'    # ← Add any extra flags

!python {METHOD_SCRIPT} \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --output-dir {DRIVE_DIR}/custom \
    {EXTRA_ARGS}


In [ ]:
# ── Custom hyperparameter sweep ───────────────────────────────────────────────
# Test different SAM rho values
for rho in [0.01, 0.05, 0.10, 0.20]:
    print(f'\n--- SAM rho={rho} ---')
    !python scripts/run_npo_sam.py \
        --model {MODEL} \
        --qlora \
        --hf-token {HF_TOKEN or ''} \
        --sam-rho {rho} \
        --no-rouge \
        --output-dir {DRIVE_DIR}/sweep_rho_{rho}

print('\n✅ Sweep complete!')


In [ ]:
# ── Run relearning attack on a specific checkpoint ────────────────────────────
CHECKPOINT = f'{DRIVE_DIR}/npo_sam/npo_sam_unlearned'  # ← change to any checkpoint

!python scripts/run_relearning_attack.py \
    --model {MODEL} \
    --qlora \
    --hf-token {HF_TOKEN or ''} \
    --checkpoint {CHECKPOINT} \
    --n-samples 50 \
    --epochs 10
